<a href="https://colab.research.google.com/github/agharikrishnan/Flyrank_ML/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/agharikrishnan/Flyrank_ML/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*I’m treating this as a ranking problem because the main goal is to decide which pages should be reviewed first. I’ll start with Logistic Regression because it is simple enough to understand and gives each page a score that can be used for ranking. I want to see whether a learned combination of the signals can improve on my Week-4 rule before trying a more complicated model.*

In [8]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

RANDOM_STATE = 42

print("Random state:", RANDOM_STATE)
print("Method: Logistic Regression")

Random state: 42
Method: Logistic Regression


## 2. Split design

*I’ll use a grouped split by client_id so pages from the same client do not end up on both sides of the split. This gives a stricter test because the model has to work on clients it did not see during training. I’ll keep this split fixed so the model and my Week-4 baseline are compared on exactly the same test pages.*

In [11]:
# Observed binary outcome:
# 1 = down, 0 = everything else
model_df["target"] = (
    model_df["trend_direction"] == "down"
).astype(int)

X = model_df[feature_cols]
y = model_df["target"]
groups = model_df["client_id"]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

train = model_df.iloc[train_idx].copy()
test = model_df.iloc[test_idx].copy()

print("Train rows:", len(train))
print("Test rows:", len(test))
print("Train clients:", train["client_id"].nunique())
print("Test clients:", test["client_id"].nunique())

overlap = set(train["client_id"]) & set(test["client_id"])
print("Client overlap:", len(overlap))

print("Train decline rate:", round(train["target"].mean(), 3))
print("Test decline rate:", round(test["target"].mean(), 3))

Train rows: 23837
Test rows: 6163
Train clients: 25
Test clients: 7
Client overlap: 0
Train decline rate: 0.55
Test decline rate: 0.511


## 3. Train + compare vs my baseline

*I’m using precision@50 because the practical question is which 50 pages should be reviewed first. I’ll compare the Logistic Regression ranking with the Week-4 rule on the same test pages. The model only gets information that would have been available at the time of the decision, and I’ll keep the random seed fixed so the result can be reproduced.*

In [12]:
from sklearn.metrics import precision_score

# Prepare train/test data
X_train = train[feature_cols].copy()
y_train = train["target"]

X_test = test[feature_cols].copy()
y_test = test["target"]

# Fill missing values using training medians only
train_medians = X_train.median()

X_train = X_train.fillna(train_medians)
X_test = X_test.fillna(train_medians)

# Logistic Regression pipeline
model = Pipeline([
    ("scale", StandardScaler()),
    ("logistic", LogisticRegression(
        max_iter=1000,
        random_state=RANDOM_STATE
    ))
])

model.fit(X_train, y_train)

# Probability of the observed "down" outcome
model_scores = model.predict_proba(X_test)[:, 1]

# -----------------------------
# Week-5 model precision@50
# -----------------------------
K = 50

model_order = np.argsort(-model_scores)
model_top_k = model_order[:K]

model_precision_at_50 = y_test.iloc[model_top_k].mean()

# -----------------------------
# Week-4-style transparent rule
# -----------------------------
baseline_test = test.copy()

baseline_test["visibility_points"] = pd.cut(
    baseline_test["impressions_90d"],
    bins=[-1, 100, 500, 1000, 5000, float("inf")],
    labels=[0, 1, 2, 3, 4]
).astype(int)

baseline_test["staleness_points"] = (
    baseline_test["days_since_last_update"].between(91, 180)
).astype(int)

baseline_test["baseline_score"] = (
    baseline_test["visibility_points"] +
    baseline_test["staleness_points"]
)

baseline_order = np.argsort(
    -baseline_test["baseline_score"].to_numpy()
)

baseline_top_k = baseline_order[:K]

baseline_precision_at_50 = (
    baseline_test["target"].iloc[baseline_top_k].mean()
)

base_rate = y_test.mean()

comparison = pd.DataFrame({
    "method": [
        "Week-4 baseline",
        "Logistic Regression"
    ],
    "precision_at_50": [
        baseline_precision_at_50,
        model_precision_at_50
    ]
})

print("Test base rate:", round(base_rate, 3))
display(comparison)

Test base rate: 0.511


,method,precision_at_50
0,Week-4 baseline,0.28
1,Logistic Regression,0.54


## 4. Errors and interpretation

*The Logistic Regression model did better than my simple Week-4 rule on the same test set. At the top 50, the model had a precision of 0.54 compared with 0.28 for the baseline. This is useful, but the improvement is not enough by itself to say that the model will always work better. I also want to check which features the model relies on and look at some of the pages it gets wrong.*

In [13]:
logistic_model = model.named_steps["logistic"]

importance = pd.DataFrame({
    "feature": feature_cols,
    "coefficient": logistic_model.coef_[0],
    "absolute_coefficient": np.abs(logistic_model.coef_[0])
})

importance = importance.sort_values(
    "absolute_coefficient",
    ascending=False
)

display(importance[["feature", "coefficient"]])

,feature,coefficient
5,content_age_days,-0.384406
3,pageviews_90d,-0.338138
1,sessions_90d,0.238756
4,days_since_last_update,0.236955
2,clicks_90d,-0.122041
6,word_count,0.112875
0,impressions_90d,0.038177
7,avg_position,-0.003769


*The largest coefficients show which signals have the strongest relationship with the model's estimated chance of a page being down. I am treating these as model signals, not proof that one feature causes a decline. I will also check individual errors because a useful model can still make mistakes on particular pages.*

In [15]:

test_results = test.copy()
test_results["model_score"] = model_scores
test_results["predicted"] = (test_results["model_score"] >= 0.5).astype(int)

false_positives = test_results[
    (test_results["predicted"] == 1) &
    (test_results["target"] == 0)
].sort_values("model_score", ascending=False)

false_negatives = test_results[
    (test_results["predicted"] == 0) &
    (test_results["target"] == 1)
].sort_values("model_score", ascending=True)

print("Three false positives:")
display(
    false_positives[
        ["content_id", "model_score", "trend_direction",
         "impressions_90d", "sessions_90d",
         "days_since_last_update", "avg_position"]
    ].head(3)
)

print("Three false negatives:")
display(
    false_negatives[
        ["content_id", "model_score", "trend_direction",
         "impressions_90d", "sessions_90d",
         "days_since_last_update", "avg_position"]
    ].head(3)
)

Three false positives:


,content_id,model_score,trend_direction,impressions_90d,sessions_90d,days_since_last_update,avg_position
10870,content_a5dbb404bdc2,0.739047,stable,79035,44,106,8.7
15621,content_dcd38075ec2c,0.731487,flat,3,2,104,6.0
22204,content_ef9bdd92b523,0.730395,flat,6,1,104,4.3


Three false negatives:


,content_id,model_score,trend_direction,impressions_90d,sessions_90d,days_since_last_update,avg_position
21565,content_9532f197bbc8,0.021279,down,309192,1098,104,2.0
21819,content_4c36c775b818,0.067703,down,463103,1114,20,2.3
17127,content_8818fd6d967f,0.182115,down,83603,841,104,3.4


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.